# Parte 3: Desenho do Experimento A/B
Nesta etapa final, documentamos o design estatístico necessário para validar se a nossa Inteligência de Priorização (Métrica de Score WLB + Decaimento + DDD) é superior à escolha determinística aleatória (ou ordem alfabética) em vigor.

In [1]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## 1. Definição de Hipóteses

*   **Grupos do Experimento:**
    *   **Grupo A (Controle):** Envio de mensagens baseado nas regras atuais (seleção de um número aleatório vinculado ao cidadão).
    *   **Grupo B (Tratamento):** Envio de mensagens baseado no `score` do novo algoritmo de priorização (selecionando o Top 1 da inteligência para cada cidadão).
*   **Hipótesese Nula ($H_0$):** $p_B \le p_A$ (A taxa de entrega no grupo tratado com o Algoritmo é menor ou igual ao Controle)
*   **Hipótese Alternativa ($H_A$):** $p_B > p_A$ (A taxa de entrega no grupo com Algoritmo de Escolha é estatisticamente **maior** do que do controle - Teste Unicaudal)

## 2. Métricas

*   **Métrica Primária (Sucesso):** OMP (Open Message Proportion) / Delivery Rate. (Contatos com `status_disparo = 'delivered'` ou status de leitura).
*   **Métricas Secundárias:**
    *   Custo por Cidadão Engajado (diminuição da taxa de tentativa e erro, diminuindo o faturamento da plataforma de envio).

## 3. Parâmetros Estatísticos (Sample Size)
Vamos supor as seguintes premissas com base na EDA histórica:
*   **Baseline (Conversão do Grupo A):** $\sim 26\%$ de entrega no primeiro disparo com escolha aleatória.
*   **Minimum Detectable Effect (MDE - Uplift):** Esperamos aumentar isso em pelo menos $4\%$ reais, totalizando $30\%$.
*   **Nível de Significância (Alpha / $\alpha$):** Promabilidade de Falsos Positivos de $5\%$ ($\alpha = 0.05$).
*   **Power / Sensibilidade ($1 - \beta$):** 80% ($0.8$).

In [2]:
import scipy.stats as stats

# Parametros
p1 = 0.26 # baseline
p2 = 0.30 # mde baseline + uplift
alpha = 0.05
power = 0.80

z_alpha = stats.norm.ppf(1 - alpha)
z_beta = stats.norm.ppf(power)

# Formula padrao 
p_pool = (p1 + p2) / 2
numerator = (z_alpha * np.sqrt(2 * p_pool * (1 - p_pool)) + z_beta * np.sqrt(p1 * (1 - p1) + p2 * (1 - p2))) ** 2
denominator = (p1 - p2) ** 2
n_per_group = numerator / denominator

print(f"Tamanho de amostra necessário por Variante: {int(np.ceil(n_per_group))} disparos (cidadãos únicos).")
print(f"Total de amostra: {int(np.ceil(n_per_group)) * 2}")

Tamanho de amostra necessário por Variante: 1557 disparos (cidadãos únicos).
Total de amostra: 3114


## 4. Duração do Experimento

Considerando que na `base_disparo_mascarado` fazemos em média alguns milhares/milhões de envios mensais, se destinarmos 10% da base de envios por dia para o experimento:

* A Prefeitura envia um total de *X* mil msgs por dia.
* Se testarmos apenas as campanhas de IPTU (para manter consistência e ter o mesmo template), e dispararmos 10% diários (cerca de 1000/dia em teste).
* Os ~4000 contatos seriam preenchidos em menos de **1 semana** de disparos ordinários controlados.

## 5. Como medir ao fim do Teste?

Usaremos o Teste Z para proporções independentes.